# RAG

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create the document object
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
]

## Manual RAG

In [5]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [6]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['1cf2b704-860c-40d3-b6ea-5c572a4d27d6',
 '3402742b-a051-4642-bcde-c3ca97371f75']

In [8]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2}
)

In [9]:
# Invoke the retriever

query = "What is LangGraph"
retrieved_docs = retriever.invoke(query)

In [10]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

"""
or
def docs_to_text(documents: list[Document])-> str:
    text = ""
    for doc in documents:
        text = text + doc.page_content + "\n\n"
    return text
"""

'\nor\ndef docs_to_text(documents: list[Document])-> str:\n    text = ""\n    for doc in documents:\n        text = text + doc.page_content + "\n\n"\n    return text\n'

In [11]:
context = format_docs(retrieved_docs)
print(context)

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [12]:
# Create the prompt

prompt = f"""
Answer the question using the following context.

Context:
{context}

Question:
{query}
"""

In [13]:
# Invoke the llm

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'Ev4GCvsGARFNMg+O5O8kciwhanrbvjHVpMWlf+1ckcOfRq5rHoHkmOoJmmCAHytXY9X7MhEpC6fqZPUOMabNVLEQH+Ac03hEn4dgxFwm2Xk/CpjHRXMk8oT6DPxyXTEiId2PG38L9t103qgf3aArOq8F1wSPyN+6vqwr+dFY6/R4uGcDl0KDOIyj7IEImViR2yjzWS6ZIrGh66rnOotg14GL+XsKDu7FWNS+Xq4norsjxqhE3mzszKkmJ7WoxPCGIx5PcMxrLvX+rhDDMbBpED/Tm8YajDTdILGyFAYa46bXsnfMkQEHinhwoniE/xqYZ+lf7vl+LUnuiilXSM1VNiZPmJQm7TiZavowSR0YGOoBXiy0SruK5j9Mb8bBKLZoMNsfGQPNfPDOL0yyqyHTIoAtDt5oxLZYnpCdCwOPW3kndShkJH0RRl0/08puyLHV5pXlJmLIosmlaa1hK1YQiVzllMoyARz+3Dshjlzp3KnHlMOefG4WRFaW44ux9ctyzBaJz0OBS0FGO98LdQm0ezbuWPalBTyYGUyi08rrEYbtqbjrq3o3N9WZEuKNQxwRD8nLejSYLHcZU9B1VUoZhVZm+IyUPOuYvOVpv94FSnr9V3kuPTTkoewfACzV7TKxbgsUJHMklS/cI3NqrPfvdrLfvGI4hY9cAfHGq9SZadm1O2zfqrjizJI7JnA92CWIdlwUoVT9AkCytIyvOcESQawzLmPMkk4lpyww6TLRNyrBBc5mNdEyE3mRQpIom875GVAc5sE3cmLz4v/v74hsdSFylI/j3UA1pfl6fZVyYL9R/xstpTByn9jNP/agA0ecy+AuDTI0OkCwV5KJ0srPOxAcjNn1VDC

# RAG Chain with Runnables

In [14]:
query = "What is LangGraph?"

In [15]:
# Convert format_docs, prompt to runnable
from langchain_core.runnables import RunnableLambda

format_docs_runnable = RunnableLambda(format_docs)

In [16]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever_chain = RunnableParallel(
    {
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough()
    }
)

In [17]:
# Create prompt
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
"""
Answer the question using the following context.

If the context does not contain enough information to answer the question, 
say that you don't have enough information.

Context:
{context}

Question:
{query}
""")

In [18]:
# Debug 
retrieved = retriever_chain.invoke(query)
print(retrieved['context'])

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [19]:
# Create RAG Chain
RAG_Chain = retriever_chain | prompt | llm

In [20]:
response = RAG_Chain.invoke(query)

In [21]:
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'ErQICrEIARFNMg/v6UcovtfmTkPM/SXM6y3rMZ0NJUr34OlYp5eju3z5yN7ApY0smhSN/bTQDFXGJhpldqx1G+LVxzayjygKc9XFACZHU9aiTWGz9CFAlQsaCrf4F6ReWWxYov7dD6d2km8pXM280n2o6KCzhOBTysFDrvnYAXQuF4vNP3qTL7fcrhIwJqNpmGuVosl87EK3DyZ6DsnukghJKXgux0NgNZmHa/q+hv7DRcf69tVLIJqSnEKm5UXLn1xIKNIKuB5nwf7GrPPODy7C53X6gsduWD/YcVv0FoJHJ20IOpRD8PLpXflMR8ls+VYr+keGgR+gJOjHo8L7pA2OH4fuUt8lHZdpGCc+jdsds4z7eG6RPUq3Ys0Brw6vCvnBmj195S15ojh00ojyoUDqp4Eq2wGkZvoVjRNbgppYaa6Ce+uGJuSEmoNeHZVmB234vrBMgUQUU3Oa6UcumYzXM/wx9NO1QJHpVfVwEVpiv6kV1fK0VXeB6WQZGidJskyQ6FIGZe6vD+A1uBDtEk30lwipyNU5wf0TyyNBYRTH/OBaW3+8/CEBvXa3v+Nc+ip5ap2zDvGyx6j9t+dK4y6HEF0X4nV/bfsYiSzHMWnQkBnpoe+mG6T4OzkiXFT9j8fCsk0epwaJ8m4/SsO0FE9xhyV+WALP234QBeCiu5uUXACQsTPdFl7cKKTqKu3wIhC1RpDx89HiYSZrY4HlBlIg0bUdTnS8CDqoKgHAirVWhhLvtFOJsKaRkm4PWcOE7BkZvBoaq8H4rRc0pLn8H2KEinuRwfmgm1dHrXg5ZaRUK7NdOM9W/MEZxQ3VdG7bow83tMN0tDhoafsiQ2rHE5zTIE5BiYI

RAG 2 major stages:

```text
                    RAG
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
     Retrieval stage       Generation stage
          │                     │
     Find relevant docs    Generate answer
```

In [22]:
response = RAG_Chain.invoke("What is the newest AMD GPU chip")

In [23]:
print(response.content)

[{'type': 'text', 'text': "I don't have enough information.", 'extras': {'signature': 'EtMHCtAHARFNMg9zSn1pPfxpdM99ee5DwGdw6FqGezqNPJzg91T6PsKCp4zs0cwNvVZC4dYl6BqJ5xfBsEe9RuDsmdd1HjsGal9JSc6qkszAI7sYr3j4Km5UFhPx0LFYfJyXcxpPiVPqNmGvU9MY/KcIFAlGaQE+EcdhI17L/HGvw5dC3+z/ZE/HzUbtP6afwi2ug7nVkr1eHU//QqqZT+cxwiKLngEM4lmY1yS1ZI8PFOXWIMDSo1ZGlBGzcM0JQkwr8QtNenEdCWZzIJ5fIupzstWTAT4Trvm6iiGwNRLmFJDDYo9HdZ/oYeVDpPhNTMIxzDagbf80x1fxhjUFO90XG0Vmez1xsSWlaVKtG7pMbMYlJWhFz4jIXnkkOLuihknDhsDHFGmpg2Rx9lfm+/OUQB/YR3bOTdQ2B1cTv5KupC2rNsQ5iw0L6Yz6yDRQvMM6Q7jSUiA7FtLcGdHqVzGKW4Jtcs34siaB+h/68gQz10Of4hnphZtkIzTaZTSQgO/asd8UF4FVenDwhZfZ1qJNj9Yae0EmQGNO5GSgNao+xagENUl9JiGDxA0jPytbU1T3fl6lvcja9smrkf4W371cB7noPjwi/VM0sUfu/G+N/7bZL7tadHzM/sdNt4kATtLDJ+onopoKzdmpts8Q2Hk14hJ6SRf7HBYtSuUZEnYRDT7LO5UZpPjxiJJO2cBGWan10jFbfoc3McM9zyxX4OaJZ0EumZ4seUZTrq9ZZpEQCaBqcsOZJ+j3R5DNrpyI1xY2JvKk9UBDYRqO1QpsZRx4yHprxjJF6RqQtZpfAmKE8GwDmeA0Xl6tUnKxWSSigsdif2doVxrvmHWHKSgiSt+kxA1f13NjFajeUUS+gAxDlaZrOhBghzn45bXshVSjAMb/bSHtYxWNDpEzC

## Advanced RAG pipeline

In [24]:
# Create the document objects
from langchain_core.documents import Document

documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
    Document(
        page_content="Retrievers are components that return relevant documents for a given query.",
        metadata={"source": "retrievers.txt"}
    ),
    Document(
        page_content="Vector stores are used to store and search vector representations of documents.",
        metadata={"source": "vector_stores.txt"}
    ),
    Document(
        page_content="Embeddings represent text as numerical vectors that capture semantic relationships.",
        metadata={"source": "embeddings.txt"}
    ),
    Document(
        page_content="RAG combines information retrieval with language generation to provide context to an LLM.",
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content="LCEL allows LangChain components to be composed into executable pipelines using the pipe operator.",
        metadata={"source": "lcel.txt"}
    ),
    Document(
        page_content="RunnableParallel allows multiple Runnable components to execute using the same input.",
        metadata={"source": "runnable_parallel.txt"}
    ),
    Document(
        page_content="RunnablePassthrough forwards the original input without modifying it.",
        metadata={"source": "runnable_passthrough.txt"}
    ),
    Document(
        page_content="RunnableLambda converts a Python function into a Runnable component.",
        metadata={"source": "runnable_lambda.txt"}
    ),
    Document(
        page_content="Prompt templates provide a reusable structure for constructing prompts dynamically.",
        metadata={"source": "prompt_templates.txt"}
    ),
    Document(
        page_content="Structured output allows an LLM response to follow a predefined schema.",
        metadata={"source": "structured_output.txt"}
    ),
    Document(
        page_content="Pydantic models can be used to define and validate structured data returned by an LLM.",
        metadata={"source": "pydantic.txt"}
    ),
    Document(
        page_content="Tool calling allows an LLM to request that an external function or system be used.",
        metadata={"source": "tool_calling.txt"}
    ),
    Document(
        page_content="A tool call contains the name of the requested tool and the arguments generated by the LLM.",
        metadata={"source": "tool_calls.txt"}
    ),
    Document(
        page_content="The application executes a tool after receiving a tool call from the LLM.",
        metadata={"source": "tool_execution.txt"}
    ),
    Document(
        page_content="Tool descriptions help an LLM understand when a tool should be selected and how it should be used.",
        metadata={"source": "tool_descriptions.txt"}
    ),
    Document(
        page_content="Similarity search retrieves documents whose vector representations are close to the query vector.",
        metadata={"source": "similarity_search.txt"}
    ),
    Document(
        page_content="Maximum Marginal Relevance can improve retrieval diversity by balancing relevance and similarity between retrieved documents.",
        metadata={"source": "mmr.txt"}
    ),
    Document(
        page_content="Metadata filtering allows retrieval results to be restricted using attributes associated with documents.",
        metadata={"source": "metadata_filtering.txt"}
    ),
]

In [ ]:
# Add documents to the vector_store

vector_store.add_documents(documents)

In [26]:
# Create the retriever

retriever = vector_store.as_retriever(
    search_type ="mmr",
    search_kwargs = {
        "k":3,
        "fetch_k": 10
        }
)

In [27]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    docs = "\n\n".join(
        f"Page content: {doc.page_content}\n Page_source: {doc.metadata}"
        for doc in documents
        )
    return docs
    


In [28]:
# Convert the format function to runnable
format_docs_runnable = RunnableLambda(format_docs)

In [29]:
# Create history
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content=query),
    AIMessage(content=response.content[0]['text'])
]



In [44]:
print(history)

[HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}), AIMessage(content="I don't have enough information.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [30]:
# Create history
"""from langchain_core.messages import HumanMessage, AIMessage

def history (query, previous_query, previous_response):
    history = [
        HumanMessage(content=query),
        AIMessage(content=response.content[0]['text'])
    ]
    return(query, history)

"""

"from langchain_core.messages import HumanMessage, AIMessage\n\ndef history (query, previous_query, previous_response):\n    history = [\n        HumanMessage(content=query),\n        AIMessage(content=response.content[0]['text'])\n    ]\n    return(query, history)\n\n"

In [31]:
retriever_chain = RunnableParallel({
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough(),

})

In [32]:
query2 = "what is the use case of it?"

In [49]:
# Query rewriter 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
query_rewrite_prompt = ChatPromptTemplate([
    (
        "system",
        "Rewrite the user's latest question into a standalone question "
        "that can be understood without the conversation history. "
        ),
        MessagesPlaceholder("history"),
        ("human", "{query}")
])

In [88]:
from langchain_core.output_parsers import StrOutputParser
query_rewriter = query_rewrite_prompt | llm 
# query_rewriter = query_rewrite_prompt | llm | StrOutputParser()

In [94]:
rewritten_query = query_rewriter.invoke({
    "history": history,
    "query": query2
})


In [95]:
rewritten_query_str = rewritten_query.content[0]["text"]

In [96]:
retrieved_docs = retriever_chain.invoke(rewritten_query_str)
print(retrieved_docs)

{'context': "Page content: LangGraph is designed for stateful agent workflows.\n Page_source: {'source': 'langgraph.txt'}\n\nPage content: Embeddings represent text as numerical vectors that capture semantic relationships.\n Page_source: {'source': 'embeddings.txt'}\n\nPage content: The application executes a tool after receiving a tool call from the LLM.\n Page_source: {'source': 'tool_execution.txt'}", 'query': 'What are the use cases of LangGraph?'}


In [97]:
# Create the prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using the provided context. If the context does not contain enough information to answer the question, say that you don't have enough information."),
        MessagesPlaceholder("history"),
        ("human", "Question: {query}, \nContext: {context}"),
    ]
)

In [98]:
prompt__history = prompt.invoke(
    {
        "history":history,
        "context":retrieved_docs,
        "query":query2
    }
)

In [99]:
RAG_response =  llm.invoke(prompt__history)

In [100]:
print(RAG_response)

content=[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EoYKCoMKARFNMg92O3iz098PNHlvPhQoKFXrW05WcseLZoc7M0u9RGrjbo92EkYaODE0n1kEioiWGEoFha0Cc6NtLvNxykNcFhVxsAqd3RGuyv+acfqbQ9y8NPAlnrJcqu92MivUBXfq7INRctailun2l5dBTwKosBrgvGm02nvDH3RzKKFcNu0T98DJbQv5FKWEkfa0V0VOKWzmEOrlFt/NXFfhBIG8ln6wz7rPNiI5I5kGKixucAHNSXeJzGNZwpvcgNmMXSoaDyLZafLJ6SWbGrFM88JSP8NB1GH3oEv3q5z1KoCUCPdPyOR9dflmxMNFsiFU+H+OYNYfIgbAQc+UVxCI5dEmuhtQLfJRgzvQJC1HjFSrPUsbx1O6rpiFMMW4lVXgrniUgCsEM2MtiUospwWt59SabFOuiFwrdrdi0/y84Tfx+VV2q4Ka4Mvq3aXagBPj1EYEvHu6Aa2AJuQx8yVsE0cPA+OZzhi3v10UyrzdNpt+o1Bb7UUclI4VAUbxJmF0w5GEdhrsXg4qYZwFRkEt5V5R2N2NF5B0+Aqt0vMhSSzzkWkCaKVpM8ldxE3F3j+ZioBj6NS0XX9tXmguUqsqTmB3RgiW55hhKxKnOgwpRqQ7OZQI+TWHeHK1lUxY9+A8ZbQPDshw9llAwiLb2Qhyzvy2QLTplftQAucrnWWK19FGlx4IcjMkvMsnSKKREa00X+kau8W2bZJBtjwX23BgrCkCoHpmwFx36ILB78T2FPEjk5CVV9POGBZKkiffmXGJgX8fep792GIA9yHop8HyyBIjhrFfkWZw7ENBCFNoSPCOth6A0q1rNfva5YdgtHMW47E6HWddMUK/bHp

Work Flow:
```text
Conversation history
        ↓
Query Rewriter
        ↓
Standalone query
        ↓
Retriever (MMR)
        ↓
Relevant documents
        ↓
Formatted context
        ↓
History + current question + context
        ↓
LLM
        ↓
Answer
```